# Question 1. Define the Search Tool

In [254]:
import requests

In [255]:
query = "capybara"
url = "https://en.wikipedia.org/w/api.php"

In [274]:

def search_wikipedia(query: str):
    
    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json",
        "origin": "*"
    }

    headers = {
        "User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"
    }

    response = requests.get(endpoint, params=params, headers=headers)

    # Ensure JSON
    if "application/json" not in response.headers.get("Content-Type", ""):
        print("Non‑JSON response received:")
        print(response.text[:500])
        return []

    data = response.json()
    return data["query"]["search"]



In [275]:
# How many total results did the search return?
results = search_wikipedia("capybara")
len(results)


10

# Question 2. Analyzing Search Results

In [138]:
# How many of the results contain the word "capybara"

In [276]:
word = "capybara"
cap_count = sum(word in item["title"].lower() for item in results)
cap_count


6

# Question 3. Analyzing Search Results

In [277]:
url = "https://en.wikipedia.org/w/index.php?title=Capybara&action=raw"

In [278]:
url

'https://en.wikipedia.org/w/index.php?title=Capybara&action=raw'

In [279]:

def get_page(url: str):
        
    
    headers = {
        "User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"
    }
    
      
    response = requests.get(url, headers=headers)
    data = response.content
    return data[:45000]
    

In [280]:
content = get_page(url)

In [281]:
len(content)

44449

#  Agent Set up

In [282]:
import pydantic_ai
from pydantic_ai import Agent, RunContext


In [283]:
instructions = """
You're a Wikipedia agent. 

Given a query you will search Wikipedia and return the serach result. Use the available tools as needed

Use only facts from the Wikipedia  when answering.
IMPORTANT: if you cannot find the answer, inform the user.
"""

In [284]:
import dotenv
dotenv.load_dotenv()

True

In [285]:
agent_tools = [search_wikipedia, get_page]

In [286]:
search_agent = Agent(
    name='search',
    model='openai:gpt-5-mini',
    instructions=instructions,
    tools=agent_tools
)

# Question 5. Testing Your Agent - Single Page

In [287]:
query = "What is this page about? https://en.wikipedia.org/wiki/Capybara"

In [288]:
result = await search_agent.run(query)

In [291]:
print(result.output)

That Wikipedia page is about the capybara (Hydrochoerus hydrochaeris). Key facts on the page:

- The capybara is the largest living rodent.  
- It is a large, semi‑aquatic, social, herbivorous rodent native to South America (occurring in the countries of that continent).  
- It belongs to the genus Hydrochoerus and the cavy family (Caviidae).


In [292]:
messages = result.all_messages()

In [294]:
for m in messages:
    print(m.kind)
    for p in m.parts:
        part_kind = p.part_kind
        if part_kind == 'user-prompt':
            print('USER:', p.content)
        if part_kind == 'tool-call':
            print('TOOL CALL:', p.tool_name, p.args)
        if part_kind == 'tool-return':
            print('TOOL RETURN:', p.tool_name)
        if part_kind == 'text':
            print(p.content)
    print()

request
USER: What is this page about? https://en.wikipedia.org/wiki/Capybara

response
TOOL CALL: get_page {"url":"https://en.wikipedia.org/wiki/Capybara"}

request
TOOL RETURN: get_page

response
TOOL CALL: search_wikipedia {"query":"Capybara"}

request
TOOL RETURN: search_wikipedia

response
That Wikipedia page is about the capybara (Hydrochoerus hydrochaeris). Key facts on the page:

- The capybara is the largest living rodent.  
- It is a large, semi‑aquatic, social, herbivorous rodent native to South America (occurring in the countries of that continent).  
- It belongs to the genus Hydrochoerus and the cavy family (Caviidae).



# Question 6. Testing Your Agent - Search Then Fetch

In [295]:
query = "What are the main threats to capybara populations?"

In [296]:
result = await search_agent.run(query)

In [297]:
print(result.output)

According to the Capybara article on Wikipedia, the main threats are:
- Hunting (for meat and hide) and other exploitation by people.  
- Habitat loss and fragmentation from human activities (agriculture, development).  
- Persecution where they are considered agricultural pests, and local threats such as vehicle collisions.  
- Natural predation by large predators (e.g., jaguars, caimans, anacondas) affects individuals locally.

Wikipedia also notes that, overall, the species is widespread and assessed as Least Concern on the IUCN Red List.


In [298]:
messages = result.all_messages()

In [299]:
for m in messages:
    print(m.kind)
    for p in m.parts:
        part_kind = p.part_kind
        if part_kind == 'user-prompt':
            print('USER:', p.content)
        if part_kind == 'tool-call':
            print('TOOL CALL:', p.tool_name, p.args)
        if part_kind == 'tool-return':
            print('TOOL RETURN:', p.tool_name)
        if part_kind == 'text':
            print(p.content)
    print()

request
USER: What are the main threats to capybara populations?

response
TOOL CALL: search_wikipedia {"query":"capybara threats"}

request
TOOL RETURN: search_wikipedia

response
TOOL CALL: search_wikipedia {"query":"Capybara Hydrochoerus hydrochaeris Wikipedia"}

request
TOOL RETURN: search_wikipedia

response
TOOL CALL: get_page {"url":"https://en.wikipedia.org/wiki/Capybara"}

request
TOOL RETURN: get_page

response
TOOL CALL: search_wikipedia {"query":"Capybara conservation threats hunting habitat loss Wikipedia capybara threats hunting habitat loss 'Capybara threats' 'Capybara conservation'"}

request
TOOL RETURN: search_wikipedia

response
TOOL CALL: search_wikipedia {"query":"Capybara predators 'Capybara' predators 'Hydrochoerus hydrochaeris' Wikipedia predators capybara jaguar anaconda caiman puma"}

request
TOOL RETURN: search_wikipedia

response
TOOL CALL: search_wikipedia {"query":"Hydrochoerus hydrochaeris IUCN 'Least Concern' 'capybara hunted meat hide' site:en.wikipedia